In [ ]:
#evaluating how well Graph Neural Networks (specifically Node2Vec and Unsupervised GraphSAGE) can detect money laundering by analyzing the pure structural patterns of transaction networks
#  (like fan-outs, scatter-gathers, and cycles). You are converting tabular transaction data into a PyTorch Geometric graph, generating 64-dimensional node embeddings,
#  and then using an Isolation Forest to catch anomalies based on those structural representations, finally evaluating the results using PR-AUC and dynamic budget thresholds.
import pandas as pd 
import numpy as np
import torch

print("\n1. Loading the exact same chronological subsample.")
df = pd.read_csv('HI_Small_1M_Chronological.csv')

print(f"Data loaded successfully! Shape : {df.shape}")

print("\n2. Extracting unique nodes (bank accounts)")
senders = df['Account'].unique()
receivers = df['Account.1'].unique()
unique_accounts = np.unique(np.concatenate((senders, receivers)))

print(f"Total Unique Accounts (Nodes) found: {len(unique_accounts)}")

print("\n3. Mapping Accounts to Node IDs")
account_to_id = {account: i for i, account in enumerate(unique_accounts)}

df['Sender_Node_ID'] = df['Account'].map(account_to_id)
df['Receiver_Node_ID'] = df['Account.1'].map(account_to_id)

print("Node ID mapping completed. Sample:")
print(df[['Account','Sender_Node_ID','Account.1','Receiver_Node_ID']].head())

print("\n4. Constructing the Edge List")
# Row 0 = source nodes (senders), Row 1 = target nodes (receivers)
source_nodes = df['Sender_Node_ID'].values
target_nodes = df['Receiver_Node_ID'].values

edge_index = torch.tensor(np.array([source_nodes, target_nodes]), dtype=torch.long)
print(f"Edge index shape: {edge_index.shape}")

#Graph Initialization and Node Mapping
#This block transforms raw transaction data into a formal network structure by identifying every unique bank account and mapping it to a contiguous numerical ID.
#It then constructs the edge_index tensor, a PyTorch Geometric requirement that maps the directional flow of funds from sender nodes to receiver nodes. 
#This preprocessing is crucial because Graph Neural Networks require continuous numerical matrices, rather than alphanumeric account strings, to process the structural topology
#of the financial network.




1. Loading the exact same chronological subsample.
Data loaded successfully! Shape : (1000000, 11)

2. Extracting unique nodes (bank accounts)
Total Unique Accounts (Nodes) found: 423676

3. Mapping Accounts to Node IDs
Node ID mapping completed. Sample:
     Account  Sender_Node_ID  Account.1  Receiver_Node_ID
0  800F116A0           19688  800F116A0             19688
1  800D856D0           17665  800D856D0             17665
2  80A464B90          214639  80A464B90            214639
3  8008C1090           11415  801A97310             34685
4  800D73230           17558  800D73230             17558

4. Constructing the Edge List
Edge index shape: torch.Size([2, 1000000])


In [ ]:
from torch_geometric.data import Data
print("\n5. Creating Structural Node Features")
num_nodes = len(unique_accounts)
node_features = torch.ones((num_nodes, 1), dtype=torch.float)  
#basically to create a simple feature matrix where every one of the nodes (423676) has a single feature of '1'"firstdraftcodetabular copy.ipynb"
#that way the graph models are forced to find criminals based purely on network structures (cycles and fan-outs)

print("\n6. Building the PyG Data Object")
graph_data = Data (x=node_features, edge_index=edge_index)
# to check but i think data abject is the equivalent of dataframe in pytorch geometric, it contains all the info about the graph (nodes, edges, features) and can be used as input for graph neural network models

print("\n---- Final Graph Data Object ----")
print(graph_data)

#quick check but you can delete later 
print(f"\nIs the graph valid? {graph_data.validate()}")
print(f"Number of nodes: {graph_data.num_nodes}")
print(f"Number of edges: {graph_data.num_edges}")

#Feature Matrix and Graph Object Assembly
#a structural feature matrix where every node is assigned a uniform, "dummy" value of 1. This deliberate design choice forces the subsequent Graph Neural Networks to identify illicit behavior based entirely on 
#network topology—such as cycles and scatter-gather patterns—rather than relying on account-specific metadata. Finally, it packages these features and the previously built edge list into a PyTorch Geometric Data object,
#which serves as the validated, foundational input structure for the models.



5. Creating Structural Node Features

6. Building the PyG Data Object

---- Final Graph Data Object ----
Data(x=[423676, 1], edge_index=[2, 1000000])

Is the graph valid? True
Number of nodes: 423676
Number of edges: 1000000


In [ ]:
from torch_geometric.nn import Node2Vec

print("----TRAINING NODE2VEC MODEL----")

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Hardware used for training: {device.upper()} ")

print("\n1. Initializing Node2Vec Model")

model = Node2Vec(
    graph_data.edge_index,
    embedding_dim=64, # Compressing the network structure into 64 mathematical coordinates (lowest acceptable academic standard (add source)) just large enough to capture patters 
    walk_length=10,   # Short walks to save RAM (usually 80-100, but we have memory constraints)
    context_size=5,   #model looks at 5 accounts at a time to learn relationships
    walks_per_node=5, # Number of random walks starting from each account
    num_negative_samples=1,
    p=1.0, q=1.0,     #completely unbiased random walks (p=1, q=1)
    sparse=True       # Required for memory efficiency on a laptop
).to(device)

loader = model.loader(batch_size=128, shuffle=True, num_workers=0)
optimizer = torch.optim.SparseAdam(list(model.parameters()), lr=0.01)

def train_node2vec():
    model.train()
    total_loss = 0
    for pos_rw, neg_rw in loader:
        optimizer.zero_grad()
        loss = model.loss(pos_rw.to(device), neg_rw.to(device))
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

print("\n2. Starting Training")
# Depending on your CPU, this might take a couple of minutes! Let it run.
for epoch in range(1, 11):
    loss = train_node2vec()
    print(f"Epoch {epoch:02d}, Loss: {loss:.4f}")

print("\n3. Extracting Final Unsupervised Embeddings")
model.eval()
with torch.no_grad():
    # We ask the model to output the 64-dimensional coordinates for all 423,676 accounts
    node_embeddings = model(torch.arange(graph_data.num_nodes, device=device))

print("Embeddings Extracted Successfully!")
print(f"Embeddings Shape: {node_embeddings.shape}")

#Node2Vec Initialization and Unsupervised Training
#configures and trains an unsupervised Node2Vec model to translate the network's topological structure into a continuous lower-dimensional space.
#By executing unbiased random walks through the transaction graph, the model learns the localized neighborhood and relationship context of each bank account. Finally, it outputs a 64-dimensional mathematical embedding 
#for every node, effectively compressing complex structural patterns—such as criminal fan-outs or cycles—into numerical coordinates optimized for downstream anomaly detection.


----TRAINING NODE2VEC MODEL----
Hardware used for training: CPU 

1. Initializing Node2Vec Model

2. Starting Training
Epoch 01, Loss: 2.9707
Epoch 02, Loss: 1.6751
Epoch 03, Loss: 1.1974
Epoch 04, Loss: 0.9818
Epoch 05, Loss: 0.8796
Epoch 06, Loss: 0.8286
Epoch 07, Loss: 0.8018
Epoch 08, Loss: 0.7861
Epoch 09, Loss: 0.7762
Epoch 10, Loss: 0.7694

3. Extracting Final Unsupervised Embeddings
Embeddings Extracted Successfully!
Embeddings Shape: torch.Size([423676, 64])


In [ ]:
from sklearn.ensemble import IsolationForest
from sklearn.metrics import f1_score, precision_score, recall_score
import numpy as np

print("--- ANOMALY DETECTION ON GRAPH EMBEDDINGS ---")

# 1. Converting the PyTorch embeddings back to a Numpy array for Scikit-Learn
X_graph = node_embeddings.detach().cpu().numpy()

# Exact same 'contamination' rate as the tabular model (0.1%) to ensure a fair baseline comparison
iso_forest_graph = IsolationForest(n_estimators=100, contamination=0.001, random_state=42)

print("Running Isolation Forest on 64D Network Embeddings...")
graph_predictions = iso_forest_graph.fit_predict(X_graph)

# Map to 0 (Normal) and 1 (Anomaly)
y_graph_pred_nodes = np.where(graph_predictions == -1, 1, 0)

# Attach the results back to our DataFrame
df['Graph_Guess'] = df['Sender_Node_ID'].map(lambda x: y_graph_pred_nodes[x])

# --- GLOBAL EVALUATION ---
y_true = df['Is Laundering']
y_pred = df['Graph_Guess']

f1 = f1_score(y_true, y_pred, pos_label=1)
recall = recall_score(y_true, y_pred, pos_label=1)
precision = precision_score(y_true, y_pred, pos_label=1)

print("\n--- FINAL GRAPH (NODE2VEC) GLOBAL RESULTS ---")
print(f"Minority-Class F1 Score: {f1:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")

# Dynamically calculate total criminals in this subset instead of hardcoding 575
total_criminals = y_true.sum()
criminals_caught = int(recall * total_criminals) 
print(f"Criminals Caught: {criminals_caught} out of {total_criminals}")

# --- TYPOLOGY BREAKDOWN (FOR H1 & H2) ---
print("\n--- RESULTS BY STRUCTURAL TYPOLOGY ---")

# Merge Scatter and Gather to match the thesis proposal definitions
if 'Is_Scatter' in df.columns and 'Is_Gather' in df.columns:
    df['Is_Scatter_Gather'] = df['Is_Scatter'] | df['Is_Gather']

# List the specific typologies you promised to evaluate in Section 4.2
typologies = ['Is_Fan_Out', 'Is_Scatter_Gather', 'Is_Cycle']

for typo in typologies:
    if typo in df.columns:
        # Filter the dataset to only look at this specific type of crime + normal transactions
        mask = (df[typo] == 1) | (df['Is Laundering'] == 0)
        df_subset = df[mask]
        
        y_true_sub = df_subset['Is Laundering']
        y_pred_sub = df_subset['Graph_Guess']
        
        # We use zero_division=0 to prevent warnings if a pattern hasn't appeared yet in the subset
        f1_sub = f1_score(y_true_sub, y_pred_sub, pos_label=1, zero_division=0)
        print(f"{typo.replace('Is_', '')} F1 Score: {f1_sub:.4f}")

#Anomaly Detection and Typology Evaluation (Node2Vec)
#Isolation Forest algorithm to the 64-dimensional Node2Vec embeddings to isolate anomalous accounts, maintaining a strict 0.1% contamination rate to ensure a fair baseline comparison. It then maps these node-level predictions 
#back to the transaction dataset to calculate global evaluation metrics like F1-score, Precision, and Recall. Finally, it breaks down the model's performance across specific money laundering typologies—such as fan-outs 
#and cycles—directly testing the core thesis hypotheses regarding the detection of illicit structural patterns.

--- ANOMALY DETECTION ON GRAPH EMBEDDINGS ---
Running Isolation Forest on 64D Network Embeddings...

--- FINAL GRAPH (NODE2VEC) GLOBAL RESULTS ---
Minority-Class F1 Score: 0.0282
Precision: 0.0257
Recall: 0.0313
Criminals Caught: 18 out of 575

--- RESULTS BY STRUCTURAL TYPOLOGY ---


In [ ]:
import pandas as pd

# Load the Tabular results we saved earlier (if you haven't saved them, we'll use the 1M file)
# Since we are using the same 1M Chronological file, we can just compare 'Graph_Guess' 
# against the 'Is Laundering' column which is already in your current 'df'

print(f"Total Transactions in Graph Script: {len(df)}")
print(f"Total Criminals in Dataset: {df['Is Laundering'].sum()}")
print(f"Total Flagged by Node2Vec: {df['Graph_Guess'].sum()}")

# Let's see if Node2Vec caught ANY of the 575 criminals
correct_hits = df[(df['Is Laundering'] == 1) & (df['Graph_Guess'] == 1)]
print(f"Actual Criminals caught by Node2Vec: {len(correct_hits)}")

#Reality Check and True Positive Extraction
#This block performs a straightforward reality check on the Node2Vec model's performance by comparing its structural anomaly flags directly against the dataset's ground truth labels. 
#By filtering for transactions where both the actual laundering status and the model's prediction are positive, it calculates the exact number of true positive hits. 
#This provides a quick, concrete summary of how many actual criminals were successfully identified out of the total flagged transactions.

Total Transactions in Graph Script: 1000000
Total Criminals in Dataset: 575
Total Flagged by Node2Vec: 700
Actual Criminals caught by Node2Vec: 18


In [ ]:
import torch.nn.functional as F
from torch_geometric.nn import SAGEConv
from torch_geometric.utils import negative_sampling

print("--- TRAINING UNSUPERVISED GRAPHSAGE ---")

# 1. Define the Unsupervised GraphSAGE Architecture
class UnsupervisedGraphSAGE(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super(UnsupervisedGraphSAGE, self).__init__()
        self.conv1 = SAGEConv(in_channels, hidden_channels)
        self.conv2 = SAGEConv(hidden_channels, out_channels)
        
    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = self.conv2(x, edge_index)
        return x

# 2. Initialize Model 
# Using 64 out_channels to mathematically match Node2Vec for a fair comparison!
model_sage = UnsupervisedGraphSAGE(in_channels=1, hidden_channels=32, out_channels=64).to(device)
optimizer = torch.optim.Adam(model_sage.parameters(), lr=0.01)

# 3. Training Loop with Link Prediction (Positive AND Negative Edges)
def train_sage():
    model_sage.train()
    optimizer.zero_grad()
    
    # Get embeddings using the pure structural features (graph_data.x) we built in Block 5
    z = model_sage(graph_data.x.to(device), graph_data.edge_index.to(device))
    
    # A. Score Real Transactions (Positive Edges)
    pos_edge_index = graph_data.edge_index.to(device)
    pos_out = (z[pos_edge_index[0]] * z[pos_edge_index[1]]).sum(dim=-1)
    
    # B. Score Fake Transactions (Negative Edges) to prevent Representation Collapse
    neg_edge_index = negative_sampling(
        edge_index=pos_edge_index, num_nodes=z.size(0),
        num_neg_samples=pos_edge_index.size(1)
    )
    neg_out = (z[neg_edge_index[0]] * z[neg_edge_index[1]]).sum(dim=-1)

    # C. Calculate Unsupervised Loss
    loss = F.binary_cross_entropy_with_logits(pos_out, torch.ones_like(pos_out)) + \
           F.binary_cross_entropy_with_logits(neg_out, torch.zeros_like(neg_out))
    
    loss.backward()
    optimizer.step()
    return loss.item()

print("Training GraphSAGE (This takes a moment)...")
for epoch in range(1, 11): # 10 Epochs
    loss = train_sage()
    if epoch % 2 == 0:
        print(f"Epoch {epoch:02d}, Loss: {loss:.4f}")

# 4. Extract Embeddings
model_sage.eval()
with torch.no_grad():
    sage_embeddings = model_sage(graph_data.x.to(device), graph_data.edge_index.to(device)).cpu().numpy()

print("\nGraphSAGE Embeddings Ready!")
print(f"Embeddings Shape: {sage_embeddings.shape}")

#Unsupervised GraphSAGE Architecture and Link Prediction Training
#This block defines and initializes an unsupervised GraphSAGE model, intentionally configuring it with 64-dimensional outputs to ensure a mathematically fair comparison against the Node2Vec baseline. 
#The network is trained using a link prediction objective that scores actual transactions against randomly generated fake transactions (negative sampling), a technique critical for preventing representation collapse 
#during learning. Finally, the optimized model aggregates localized neighborhood structures to generate a dense, 64-dimensional embedding matrix for all accounts, perfectly formatting the data for the upcoming anomaly detection 
#phase.

--- TRAINING UNSUPERVISED GRAPHSAGE ---
Training GraphSAGE (This takes a moment)...
Epoch 02, Loss: 7.3413
Epoch 04, Loss: 2.5180
Epoch 06, Loss: 1.6281
Epoch 08, Loss: 1.6355
Epoch 10, Loss: 1.7914

GraphSAGE Embeddings Ready!
Embeddings Shape: (423676, 64)


In [ ]:
from sklearn.ensemble import IsolationForest
from sklearn.metrics import f1_score, precision_score, recall_score
import numpy as np

print("--- PHASE 6: ANOMALY DETECTION ON GRAPHSAGE EMBEDDINGS ---")

# 1. Convert GraphSAGE embeddings to Numpy
X_sage = sage_embeddings 

# 2. Use the standard 0.1% contamination rate
iso_forest_sage = IsolationForest(n_estimators=100, contamination=0.001, random_state=42)

# Corrected print statement to 64D!
print("Running Isolation Forest on 64D GraphSAGE Embeddings...")
sage_predictions = iso_forest_sage.fit_predict(X_sage)

# 3. Map Node-level predictions back to Transactions
y_sage_pred_nodes = np.where(sage_predictions == -1, 1, 0)
df['GraphSAGE_Guess'] = df['Sender_Node_ID'].map(lambda x: y_sage_pred_nodes[x])

# --- GLOBAL EVALUATION ---
y_true = df['Is Laundering']
y_pred_sage = df['GraphSAGE_Guess']

f1_sage = f1_score(y_true, y_pred_sage, pos_label=1, zero_division=0)
recall_sage = recall_score(y_true, y_pred_sage, pos_label=1, zero_division=0)
precision_sage = precision_score(y_true, y_pred_sage, pos_label=1, zero_division=0)

print("\n--- FINAL GRAPH (GRAPHSAGE) GLOBAL RESULTS ---")
print(f"Minority-Class F1 Score: {f1_sage:.4f}")
print(f"Precision: {precision_sage:.4f}")
print(f"Recall:    {recall_sage:.4f}")

# Dynamically calculate total criminals in this subset
total_criminals = y_true.sum()
criminals_caught_sage = int(recall_sage * total_criminals) 
print(f"Criminals Caught: {criminals_caught_sage} out of {total_criminals}")

# --- TYPOLOGY BREAKDOWN (FOR H1 & H2) ---
print("\n--- RESULTS BY STRUCTURAL TYPOLOGY ---")

# List the specific typologies you promised to evaluate
typologies = ['Is_Fan_Out', 'Is_Scatter_Gather', 'Is_Cycle']

for typo in typologies:
    if typo in df.columns:
        # Filter the dataset to only look at this specific type of crime + normal transactions
        mask = (df[typo] == 1) | (df['Is Laundering'] == 0)
        df_subset = df[mask]
        
        y_true_sub = df_subset['Is Laundering']
        y_pred_sub = df_subset['GraphSAGE_Guess']
        
        f1_sub = f1_score(y_true_sub, y_pred_sub, pos_label=1, zero_division=0)
        print(f"{typo.replace('Is_', '')} F1 Score: {f1_sub:.4f}")

#Anomaly Detection and Typology Evaluation (GraphSAGE)
#This block applies an Isolation Forest algorithm to the newly generated GraphSAGE embeddings to isolate anomalous accounts, maintaining the strict 0.1% contamination rate for a direct comparison against the Node2Vec baseline. 
#It maps these node-level risk predictions back to the transaction dataset to compute global evaluation metrics such as F1-score, Precision, and Recall. Finally, it evaluates the model's performance across specific money 
#laundering typologies—such as fan-outs and cycles—providing targeted evidence to directly address the thesis hypotheses regarding structural detection.

--- PHASE 6: ANOMALY DETECTION ON GRAPHSAGE EMBEDDINGS ---
Running Isolation Forest on 64D GraphSAGE Embeddings...

--- FINAL GRAPH (GRAPHSAGE) GLOBAL RESULTS ---
Minority-Class F1 Score: 0.0000
Precision: 0.0000
Recall:    0.0000
Criminals Caught: 0 out of 575

--- RESULTS BY STRUCTURAL TYPOLOGY ---


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import SAGEConv
from torch_geometric.utils import negative_sampling
from sklearn.ensemble import IsolationForest
from sklearn.metrics import f1_score, precision_score, recall_score
import numpy as np

print("--- PHASE 7: TRAINING UNSUPERVISED GRAPHSAGE (LINK PREDICTION) ---")

# 1. Define the Unsupervised GraphSAGE Architecture
class UnsupervisedGraphSAGE(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super(UnsupervisedGraphSAGE, self).__init__()
        self.conv1 = SAGEConv(in_channels, hidden_channels)
        self.conv2 = SAGEConv(hidden_channels, out_channels) # 64D to match Node2Vec!

    def forward(self, x, edge_index):
        x = F.relu(self.conv1(x, edge_index))
        x = self.conv2(x, edge_index)
        return x

# 2. Setup Training
# Using graph_data.x (the dummy 1s) to force pure structural learning
model_unsup = UnsupervisedGraphSAGE(in_channels=1, hidden_channels=32, out_channels=64).to(device)
optimizer = torch.optim.Adam(model_unsup.parameters(), lr=0.01)

def link_prediction_loss(z, pos_edge_index):
    # Score real transactions
    pos_out = (z[pos_edge_index[0]] * z[pos_edge_index[1]]).sum(dim=-1)
    
    # Sample and score fake transactions
    neg_edge_index = negative_sampling(
        edge_index=pos_edge_index, num_nodes=z.size(0),
        num_neg_samples=pos_edge_index.size(1)
    )
    neg_out = (z[neg_edge_index[0]] * z[neg_edge_index[1]]).sum(dim=-1)

    # Calculate loss
    loss = F.binary_cross_entropy_with_logits(pos_out, torch.ones_like(pos_out)) + \
           F.binary_cross_entropy_with_logits(neg_out, torch.zeros_like(neg_out))
    return loss

print("Training GraphSAGE (This takes a moment)...")
for epoch in range(1, 11): # 10 Epochs for testing subset
    model_unsup.train()
    optimizer.zero_grad()
    
    z = model_unsup(graph_data.x.to(device), graph_data.edge_index.to(device))
    loss = link_prediction_loss(z, graph_data.edge_index.to(device))
    loss.backward()
    optimizer.step()
    
    if epoch % 2 == 0:
        print(f"Epoch {epoch:02d}, Loss: {loss:.4f}")

# 3. Extract Embeddings
model_unsup.eval()
with torch.no_grad():
    z_final = model_unsup(graph_data.x.to(device), graph_data.edge_index.to(device)).cpu().numpy()

print("\n--- PHASE 8: ISOLATION FOREST ON GRAPHSAGE EMBEDDINGS ---")

# 4. Anomaly Detection (using the exact same 0.001 contamination)
iso_forest_sage = IsolationForest(n_estimators=100, contamination=0.001, random_state=42)

print("Running Isolation Forest on 64D GraphSAGE Embeddings...")
sage_predictions = iso_forest_sage.fit_predict(z_final)

# Map Node-level predictions back to Transactions
y_sage_pred_nodes = np.where(sage_predictions == -1, 1, 0)
df['GraphSAGE_Guess'] = df['Sender_Node_ID'].map(lambda x: y_sage_pred_nodes[x])

# --- GLOBAL EVALUATION ---
y_true = df['Is Laundering']
y_pred_sage = df['GraphSAGE_Guess']

f1_sage = f1_score(y_true, y_pred_sage, pos_label=1, zero_division=0)
recall_sage = recall_score(y_true, y_pred_sage, pos_label=1, zero_division=0)
precision_sage = precision_score(y_true, y_pred_sage, pos_label=1, zero_division=0)

print("\n--- FINAL GRAPH (GRAPHSAGE) GLOBAL RESULTS ---")
print(f"Minority-Class F1 Score: {f1_sage:.4f}")
print(f"Precision: {precision_sage:.4f}")
print(f"Recall:    {recall_sage:.4f}")

total_criminals = y_true.sum()
criminals_caught_sage = int(recall_sage * total_criminals) 
print(f"Criminals Caught: {criminals_caught_sage} out of {total_criminals}")

# --- TYPOLOGY BREAKDOWN (FOR H1 & H2) ---
print("\n--- RESULTS BY STRUCTURAL TYPOLOGY ---")

typologies = ['Is_Fan_Out', 'Is_Scatter_Gather', 'Is_Cycle']

for typo in typologies:
    if typo in df.columns:
        mask = (df[typo] == 1) | (df['Is Laundering'] == 0)
        df_subset = df[mask]
        
        y_true_sub = df_subset['Is Laundering']
        y_pred_sub = df_subset['GraphSAGE_Guess']
        
        f1_sub = f1_score(y_true_sub, y_pred_sub, pos_label=1, zero_division=0)
        print(f"{typo.replace('Is_', '')} F1 Score: {f1_sub:.4f}")

#End-to-End GraphSAGE Pipeline and Evaluation
#This block consolidates the complete Unsupervised GraphSAGE workflow, first training the network via link prediction to generate 64-dimensional structural embeddings for every account. It then immediately applies an 
#Isolation Forest to these generated coordinates to flag anomalies, facilitating a seamless evaluation of the model's overall accuracy and its specific ability to catch structural laundering typologies like cycles and fan-outs.

--- PHASE 7: TRAINING UNSUPERVISED GRAPHSAGE (LINK PREDICTION) ---
Training GraphSAGE (This takes a moment)...
Epoch 02, Loss: 7.6970
Epoch 04, Loss: 2.1115
Epoch 06, Loss: 2.5891
Epoch 08, Loss: 2.6107
Epoch 10, Loss: 2.2765

--- PHASE 8: ISOLATION FOREST ON GRAPHSAGE EMBEDDINGS ---
Running Isolation Forest on 64D GraphSAGE Embeddings...

--- FINAL GRAPH (GRAPHSAGE) GLOBAL RESULTS ---
Minority-Class F1 Score: 0.0000
Precision: 0.0000
Recall:    0.0000
Criminals Caught: 0 out of 575

--- RESULTS BY STRUCTURAL TYPOLOGY ---


In [ ]:
from sklearn.metrics import precision_recall_curve, auc
import numpy as np

print("--- PHASE 9: THE THRESHOLD INVESTIGATION & PR-AUC ---")

# 1. Get raw anomaly scores from the GraphSAGE Isolation Forest
# By default, Isolation Forest gives lower/negative scores to anomalies. 
# We multiply by -1 so that HIGHER scores = HIGHER fraud risk.
raw_anomaly_scores = -iso_forest_sage.decision_function(z_final)

# 2. Map the node-level risk scores back to the actual transactions
df['GraphSAGE_Risk_Score'] = df['Sender_Node_ID'].map(lambda x: raw_anomaly_scores[x])

y_true = df['Is Laundering']
y_scores = df['GraphSAGE_Risk_Score']

# 3. Calculate the Precision-Recall Curve (AUC-PR)
# This perfectly aligns with Section 5.2 of the Literature Review!
precision_curve, recall_curve, thresholds = precision_recall_curve(y_true, y_scores, pos_label=1)
pr_auc = auc(recall_curve, precision_curve)

print(f"GraphSAGE Area Under the PR Curve (AUC-PR): {pr_auc:.4f}")

# 4. The "Real World" AML Strategy: Custom Daily Alert Budget
# Let's say a bank only has the budget to investigate 1,000 transactions today.
budget = 1000
# We find the score threshold that perfectly isolates the top 1,000 riskiest transactions
dynamic_threshold = np.sort(y_scores)[-budget]

print(f"\nBank Budget Scenario: Investigating only the top {budget} riskiest transactions.")
print(f"Dynamic Threshold cutoff set to: {dynamic_threshold:.4f}")

# Apply the custom threshold
df['Dynamic_Budget_Guess'] = (df['GraphSAGE_Risk_Score'] >= dynamic_threshold).astype(int)

# Check how many actual criminals were in that top 1,000
budget_criminals_caught = df[(df['Is Laundering'] == 1) & (df['Dynamic_Budget_Guess'] == 1)].shape[0]

print(f"Actual Criminals Caught in the Top {budget}: {budget_criminals_caught}")

#PR-AUC Evaluation and Practical Budget Simulation
#This block finalizes the model's evaluation by calculating the Precision-Recall Area Under the Curve (PR-AUC), providing a robust performance metric optimized for highly imbalanced anomaly detection. 
#It then translates these mathematical risk scores into a realistic Anti-Money Laundering (AML) strategy by applying a dynamic threshold tied to a bank's daily investigation budget. By isolating the top 
#1,000 riskiest transactions, it measures exactly how many actual laundering events the model successfully identifies under practical resource constraints.

--- PHASE 9: THE THRESHOLD INVESTIGATION & PR-AUC ---
GraphSAGE Area Under the PR Curve (AUC-PR): 0.0635

Bank Budget Scenario: Investigating only the top 1000 riskiest transactions.
Dynamic Threshold cutoff set to: -0.0000
Actual Criminals Caught in the Top 1000: 72
